# Credit Card Fraud Predictive Analytics

Exploration, preprocessing, class-imbalance analysis, model training, and evaluation.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..') / 'src'))
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from fraud_detection.data import load_dataset, split_features_target
from fraud_detection.model import build_pipeline, predict_with_threshold
from fraud_detection.evaluate import evaluate_predictions

In [ ]:
df = load_dataset('../data/raw/credit_card_fraud_2026.csv')
df.shape, df.head()

In [ ]:
df['is_fraud'].value_counts().rename(index={0:'Legitimate',1:'Fraud'}), df['is_fraud'].mean()

In [ ]:
df['is_fraud'].value_counts().sort_index().plot(kind='bar', title='Class distribution')
plt.xlabel('is_fraud')
plt.ylabel('Transactions')
plt.show()

In [ ]:
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = build_pipeline(X_train)
model.fit(X_train, y_train)
y_pred, y_prob = predict_with_threshold(model, X_test, threshold=0.5)
evaluate_predictions(y_test, y_pred, y_prob)

## Threshold trade-off

Lower thresholds generally increase fraud recall while creating more false alerts. Choose a threshold using business costs and review capacity.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
rows=[]
for threshold in [0.2,0.3,0.4,0.5,0.6,0.7]:
    pred=(y_prob>=threshold).astype(int)
    rows.append({'threshold':threshold,'precision':precision_score(y_test,pred,zero_division=0),'recall':recall_score(y_test,pred),'f1':f1_score(y_test,pred)})
pd.DataFrame(rows)